# Manual retrieval check — write your own query, compare against gold

Bypasses PERCEIVE/FORMULATE/ASSESS entirely: **you** write the retrieval
query text yourself, run it directly against the corpus via the same
`Retriever` interface `agent/pipeline.py` uses, and see what comes back
against the item's gold chunk ids. No LLM anywhere in this notebook — it
never loads a model, only the retriever + corpus, so it's fast to iterate.

Workflow:
1. Pick an item (`ITEM_INDEX`) — see its question, options, and gold chunk ids.
2. Edit `MY_QUERIES` with the query text(s) you want to try.
3. Run retrieval, compare retrieved chunk ids against gold; optionally read
   the full chunk text to judge relevance.

**Kernel**: only needs `agent`'s retrieval stack — the `traffic` conda env
works, but so does any env with `pyserini`/`pyvi` installed (no torch/GPU
required, since nothing here loads a VLM).

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "agent").exists():
    ROOT = ROOT.parent  # running from notebooks/, repo root is one level up
assert (ROOT / "agent").exists(), f"can't find the agent/ package from {ROOT}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agent.corpus import Corpus
from agent.resolve import CitationIndex
from agent.retrieval.bm25 import DEFAULT_INDEX_DIR, BM25Retriever
from agent.retrieval.mock import MockRetriever
from agent.schemas import RetrievalQuery
from agent.vehicle import CORPUS_VEHICLE_TERMS

## Config — edit this cell, then run everything below

In [2]:
# --- data ---
DATA_PATH = ROOT / "law.json"
CORPUS_PATH = ROOT / "law_db.json"

# --- retriever: "bm25" (real, needs the Lucene index built once via
#     `python -m agent.build_bm25_index`) or "mock" (deterministic, no index) ---
RETRIEVER = "bm25"
INDEX_DIR = DEFAULT_INDEX_DIR
BM25_K1, BM25_B = 0.9, 0.4

K = 5              # chunks retrieved PER QUERY (raise this while exploring by hand)
AS_OF = None       # effective_date filter, e.g. "2025-01-01"

ITEM_INDEX = 0     # index into `records`

In [3]:
records = json.loads(DATA_PATH.read_text(encoding="utf-8"))
corpus = Corpus.load(CORPUS_PATH)
citation_index = CitationIndex.from_file(CORPUS_PATH)

if RETRIEVER == "bm25":
    retriever = BM25Retriever(index_dir=INDEX_DIR, k1=BM25_K1, b=BM25_B)
else:
    retriever = MockRetriever(corpus)

print(f"{len(records)} questions | corpus: {len(corpus)} chunks (v{corpus.version}) | retriever={RETRIEVER}")
print(f"valid vehicle_filter slugs: {CORPUS_VEHICLE_TERMS}")

/home/dangle/miniconda3/envs/traffic/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2414 questions | corpus: 1071 chunks (vdf6325f94e4f) | retriever=bm25
valid vehicle_filter slugs: ('xe_mo_to', 'xe_o_to', 'xe_tai', 'xe_dau_keo', 'xe_dap', 'nguoi_di_bo')


Jul 23, 2026 4:56:13 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


## Step 1 — pick an item, see its question, options, and gold

In [14]:
rec = records[7]

gold_ids = citation_index.resolve_set(rec.get("law", []))

print(f"item_id={rec.get('id')}  type_image={rec.get('type_image')}")
print(f"Q: {rec['question']}\n")
for L in ("A", "B", "C", "D"):
    marker = " <- GT" if rec.get("GT") == L else ""
    print(f"{L}. {rec[L]}{marker}")
print(f"\ngold chunk ids ({len(gold_ids)}): {sorted(gold_ids)}")

item_id=8  type_image=cong_kenh
Q: Nếu hành vi trong ảnh của người điều khiển xe máy được khoanh đỏ ở gần giữa phía bên trái ảnh gây tai nạn giao thông thì người này bị xử lý như thế nào?

A. Phạt từ 10.400.000 đồng đến 14.600.000 đồng và bị tước quyền sử dụng giấy phép lái xe từ 01 tháng đến 03 tháng.
B. Phạt tổng cộng từ 10.400.000 đồng đến 14.600.000 đồng, bị trừ 10 điểm giấy phép lái xe và không bị tước quyền sử dụng giấy phép lái xe. <- GT
C. Phạt từ 10.000.000 đồng đến 14.000.000 đồng và bị trừ 02 điểm giấy phép lái xe.
D. Phạt từ 600.000 đồng đến 800.000 đồng và không bị trừ điểm giấy phép lái xe.

gold chunk ids (4): ['ND_168_2024_D7_K10', 'ND_168_2024_D7_K13', 'ND_168_2024_D7_K2', 'ND_168_2024_D7_K3']


## Step 2 — write your own query

Fill in three things, mirroring how `agent.pipeline.formulate()` builds a
query in production — vehicle + violation(s) + penalty-type phrase:

- `MY_VEHICLE`: natural-language vehicle phrase, e.g. `"xe mô tô"`,
  `"xe ô tô"` — stated in the query text AND mapped to a `vehicle_filter`
  slug automatically (see `CORPUS_VEHICLE_TERMS` printed above).
- `MY_VIOLATIONS`: list of predicted violations, in **statute register**
  (not scene language) — one query per entry. Add more than one if the item
  involves several simultaneous violations (e.g. carrying bulky cargo AND
  causing an accident) so retrieval can find provisions for each, not just
  whichever one the lexical match happens to favor.
- `MY_PENALTY_TYPES`: which consequence-type queries to also generate, per
  violation — any of `"phạt tiền"` (fine amount), `"trừ điểm"`
  (point-deduction), `"tước quyền"` (license suspension). Leave `[]` if the
  question only asks about the behavior/lỗi itself, not a consequence.

`MY_QUERIES` is built from these automatically below — it's still a plain
list at the end, so you can append fully custom `{"text", "vehicle_filter",
"doc_boost"}` dicts to it too if you want to try something the three knobs
above don't cover.

In [ ]:
from agent.vehicle import normalize_vehicle

MY_VEHICLE = "xe mô tô"

MY_VIOLATIONS = [
    "mang vác vật cồng kềnh, chở người đứng trên yên hoặc giá đèo hàng",
    # add more statute-register violation phrases here for multi-violation items
]

MY_PENALTY_TYPES = ["phạt tiền", "trừ điểm"]  # subset of {"phạt tiền", "trừ điểm", "tước quyền"}, or []

_PENALTY_PHRASES = {
    "phạt tiền": "mức phạt tiền",
    "trừ điểm": "trừ điểm giấy phép lái xe",
    "tước quyền": "tước quyền sử dụng giấy phép lái xe",
}

_vehicle_slug = normalize_vehicle(MY_VEHICLE)
vehicle_filter = [_vehicle_slug] if _vehicle_slug else []

MY_QUERIES = []
for _violation in MY_VIOLATIONS:
    MY_QUERIES.append(
        {"text": f"{MY_VEHICLE} {_violation}".strip(), "vehicle_filter": vehicle_filter, "doc_boost": ["LUAT_36_2024"]}
    )
    for _penalty in MY_PENALTY_TYPES:
        _phrase = _PENALTY_PHRASES[_penalty]
        MY_QUERIES.append(
            {
                "text": f"{MY_VEHICLE} {_violation} {_phrase}".strip(),
                "vehicle_filter": vehicle_filter,
                "doc_boost": ["ND_168_2024"],
            }
        )

# append any fully custom queries here if the knobs above don't cover it, e.g.:
# MY_QUERIES.append({"text": "...", "vehicle_filter": [], "doc_boost": []})

for _q in MY_QUERIES:
    print(_q)


## Step 3 — run retrieval, compare against gold

In [13]:
queries = [RetrievalQuery(**q) for q in MY_QUERIES]

retrieved_by_id = {}
for q in queries:
    for chunk in retriever.retrieve(q, corpus, K, AS_OF):
        existing = retrieved_by_id.get(chunk.chunk_id)
        if existing is None or chunk.score > existing.score:
            retrieved_by_id[chunk.chunk_id] = chunk

retrieved_sorted = sorted(retrieved_by_id.values(), key=lambda c: c.score, reverse=True)
retrieved_ids = [c.chunk_id for c in retrieved_sorted]

print("queries used:")
for q in queries:
    print(f"  - {q.text!r}  (vehicle_filter={q.vehicle_filter}, doc_boost={q.doc_boost})")
print()
print(f"retrieved ({len(retrieved_ids)}): {retrieved_ids}")
print(f"gold      ({len(gold_ids)}): {sorted(gold_ids)}")

queries used:
  - 'Người điều khiển xe máy phạt tiền '  (vehicle_filter=[], doc_boost=[])

retrieved (5): ['ND_168_2024_D32_K5', 'ND_168_2024_D32_K6', 'ND_168_2024_D20_K2', 'ND_168_2024_D20_K4', 'ND_168_2024_D47_K4']
gold      (2): ['ND_168_2024_D7_K2', 'ND_168_2024_D7_K3']


## Step 4 (optional) — read the full chunk text

Useful when a retrieved chunk isn't gold (is it actually irrelevant, or a
reasonable near-miss?) or a gold chunk didn't come back at all (what does it
actually say — is the query missing the right vocabulary?).

In [7]:
print("RETRIEVED chunks:\n")
for c in retrieved_sorted:
    marker = "  <- GOLD" if c.chunk_id in gold_ids else ""
    print(f"[{c.chunk_id}] score={c.score:.3f}{marker}")
    print(f"{c.content}\n")

missing_gold = gold_ids - set(retrieved_ids)
if missing_gold:
    print("MISSED gold chunks (in gold, never retrieved):\n")
    for cid in sorted(missing_gold):
        chunk = corpus.by_id.get(cid)
        text = chunk.text if chunk else "(not in corpus!)"
        print(f"[{cid}]")
        print(f"{text}\n")

RETRIEVED chunks:

[ND_168_2024_D12_K2] score=9.401
Văn bản: Nghị định xử phạt vi phạm hành chính về trật tự, an toàn giao thông đường bộ. Trích dẫn: Khoản 2, Điều 12, 168/2024/NĐ-CP. Ngữ cảnh: CHƯƠNG II | MỤC 1. VI PHẠM QUY TẮC GIAO THÔNG ĐƯỜNG BỘ. Tên điều hoặc mục: Xử phạt, trừ điểm giấy phép lái xe các hành vi vi phạm khác về quy tắc giao thông đường bộ; sử dụng lòng đường, vỉa hè vào mục đích khác. Phương tiện: xe_mo_to, xe_o_to, xe_dap. Nội dung: 2. Phạt tiền từ 200.000 đồng đến 250.000 đồng đối với cá nhân thực hiện một trong các hành vi vi phạm sau đây:
a) Tập trung đông người trái phép, nằm, ngồi trên đường bộ gây cản trở giao thông;
b) Đá bóng, đá cầu, chơi cầu lông hoặc các hoạt động thể thao khác trái phép trên đường bộ; sử dụng bàn trượt, pa-tanh, các thiết bị tương tự trên phần đường xe chạy;
c) Điều khiển vật thể bay, tàu bay không người lái, phương tiện bay siêu nhẹ hoạt động trong phạm vi khổ giới hạn đường bộ gây cản trở hoặc nguy cơ mất an toàn cho người, phương tiện